In [1]:
import os
os.environ["HADOOP_USER_NAME"] = "root"

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField , StructType , IntegerType , StringType , DateType , DoubleType
from pyspark.sql.functions import col , trim , month , year

In [3]:
spark = SparkSession\
        .builder\
        .appName('Silver')\
        .master('local[*]')\
        .getOrCreate()

### Customers Table: Cleaning & Transformation

In [5]:
customers = spark.read.parquet("hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/parquet/customers")

In [6]:
customers.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)



In [7]:
customers = customers.na.drop()

In [8]:
customers = customers.dropDuplicates(['id'])

In [9]:
customers_columns = customers.columns

for column in customers_columns:
    null_count = customers.filter(col(column).isNull()).count()
    print(f"{column}: {null_count} null values")

id: 0 null values
name: 0 null values
email: 0 null values
country: 0 null values


In [10]:
duplicate_ids = customers.groupBy("id").count().filter(col("count") > 1)
duplicate_ids.show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [11]:
print("Total:", customers.count(), "| Distinct:", customers.distinct().count())

Total: 100 | Distinct: 100


In [12]:
customers_string_columns = [col_name for col_name, dtype in customers.dtypes if dtype == 'string']

for column in customers_string_columns:
    customers = customers.withColumn(column , trim(col(column)))

In [13]:
customers = customers.withColumnRenamed('id' , 'customer_id')\
                     .withColumnRenamed('name' , 'customer_name')

In [14]:
customers.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)



In [15]:
dim_customers = customers  

dim_customers.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/silver/dim_customers"
)

### Orders Table: Cleaning & Transformation

In [19]:
orders = spark.read.parquet("hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/parquet/orders" )

In [20]:
orders.printSchema()

root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: integer (nullable = true)
 |-- status: string (nullable = true)



In [21]:
orders = orders.na.drop()

In [22]:
orders = orders.dropDuplicates(['id'])

In [23]:
orders_columns = orders.columns

for column in orders_columns:
    null_count = orders.filter(col(column).isNull()).count()
    print(f"{column}: {null_count} null values")

id: 0 null values
customer_id: 0 null values
order_date: 0 null values
total_amount: 0 null values
status: 0 null values


In [24]:
duplicate_ids = orders.groupBy('id').count().filter(col('count') > 1)
duplicate_ids.show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [25]:
print("Total:" , orders.count() , "| Distinct:" , orders.distinct().count())

Total: 300 | Distinct: 300


In [26]:
orders = orders.filter(col('total_amount') > 0)

In [27]:
orders_string_columns = [col_name for col_name, dtype in orders.dtypes if dtype == 'string']

for column in orders_string_columns:
    orders = orders.withColumn(column , trim(col(column)))

In [28]:
orders = orders.withColumnRenamed('id' , 'order_id')\
               .withColumnRenamed('status' , 'order_status')

In [29]:
orders = orders.withColumn('order_month' , month(col('order_date')))\
               .withColumn('order_year' , year(col('order_date')))

In [30]:
dim_orders = orders

dim_orders.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/silver/dim_orders"
)

### Products Table: Cleaning & Transformation

In [31]:
products = spark.read.parquet("hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/parquet/products" )

In [32]:
products.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)



In [33]:
products = products.na.drop()

In [34]:
products = products.dropDuplicates(['id'])

In [35]:
products_columns = products.columns

for column in products_columns:
    null_count = products.filter(col(column).isNull()).count()
    print(f"{column}: {null_count} null values")

id: 0 null values
name: 0 null values
category: 0 null values
price: 0 null values


In [36]:
duplicate_ids = products.groupBy('id').count().filter(col('count') > 1)
duplicate_ids.show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [37]:
print("Total:" , products.count() , "| Distinct:" , products.distinct().count())

Total: 10 | Distinct: 10


In [38]:
products = products.filter(col('price') > 0)

In [39]:
products_string_columns = [col_name for col_name , dtype in products.dtypes if dtype == 'string']

for column in products_string_columns:
    products = products.withColumn(column , trim(col(column)))

In [40]:
products = products.withColumnRenamed('id' , 'product_id')\
           .withColumnRenamed('name' , 'product_name')\
           .withColumnRenamed('price' , 'product_price')

In [41]:
dim_products = products

dim_products.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/silver/dim_products"
)

### order item Table: Cleaning & Transformation

In [42]:
order_items = spark.read.parquet("hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/parquet/order_items")

In [43]:
order_items = order_items.na.drop()

In [44]:
order_items = order_items.dropDuplicates(['id'])

In [45]:
order_items_columns = order_items.columns

for column in order_items_columns:
    null_count = order_items.filter(col(column).isNull()).count()
    print(f"{column}: {null_count} null values")

id: 0 null values
order_id: 0 null values
product_id: 0 null values
quantity: 0 null values
unit_price: 0 null values


In [46]:
duplicate_ids = order_items.groupBy('id').count().filter(col('count') > 1)
duplicate_ids.show()

+---+-----+
| id|count|
+---+-----+
+---+-----+



In [47]:
print("Total:" , order_items.count() , "| Distinct:" , order_items.distinct().count())

Total: 600 | Distinct: 600


In [48]:
order_items = order_items.filter( (col('unit_price') > 0) &  (col('quantity') > 0))

In [49]:
order_items_string_columns = [col_name for col_name , dtype in order_items.dtypes if dtype == 'string']

for column in order_items_string_columns:
    order_items = order_items.withColumn(column , trim(col(column)))

In [50]:
order_items = order_items.withColumnRenamed('id' , 'order_item_id')

In [51]:
order_items = order_items.withColumn('total_price' , col('quantity') * col('unit_price'))

In [52]:
dim_order_items = order_items

dim_order_items.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/silver/dim_order_items"
)

### Create Fact Table

In [57]:
fact_sales = dim_order_items.alias('OI').join(dim_orders.alias('O') , 'order_id' , 'inner')\
             .select(
                 col('OI.order_item_id'),
                 col('O.order_id'),
                 col('O.customer_id'),
                 col('OI.product_id'),

                 col('O.order_date'),
                 col('O.order_month'),
                 col('O.order_year'),
                 col('O.order_status'),
                 
                 col('OI.quantity'),
                 col('OI.unit_price'),
                 col('OI.total_price')
             )

In [59]:
fact_sales.write.mode('overwrite').parquet(
    "hdfs://hadoop-namenode:9000/user/hadoop/data/Big_Data/silver/fact_sales"
)